In [5]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
import pickle as pkl

In [1]:
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# movie prediction 

In [8]:
movies=pd.read_csv('movies.csv')
movies.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [9]:
movies.shape

(4803, 24)

In [10]:
movies.isnull().sum()

index                      0
budget                     0
genres                    28
homepage                3091
id                         0
keywords                 412
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
cast                      43
crew                       0
director                  30
dtype: int64

In [11]:
# selecting the relevent features for recomendations
selected_features=['genres','keywords','tagline','cast','director']
print(selected_features)

['genres', 'keywords', 'tagline', 'cast', 'director']


In [12]:
# replacing the null values with null string

for feature in selected_features:
    movies[feature]=movies[feature].fillna('')

In [13]:
movies.isnull().sum()

index                      0
budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                    0
title                      0
vote_average               0
vote_count                 0
cast                       0
crew                       0
director                   0
dtype: int64

In [14]:
# combining all the 5 selected features
combined_features=movies['genres']+' '+movies['keywords']+' '+movies['tagline']+' '+movies['cast']+' '+movies['director']

In [15]:
print(combined_features)

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4798    Action Crime Thriller united states\u2013mexic...
4799    Comedy Romance  A newlywed couple's honeymoon ...
4800    Comedy Drama Romance TV Movie date love at fir...
4801      A New Yorker in Shanghai Daniel Henney Eliza...
4802    Documentary obsession camcorder crush dream gi...
Length: 4803, dtype: object


In [16]:
# converting the text data to feature vector
vectorizer=TfidfVectorizer()

In [17]:
feature_vector=vectorizer.fit_transform(combined_features)

In [18]:
print(feature_vector)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 124266 stored elements and shape (4803, 17318)>
  Coords	Values
  (0, 201)	0.07860022416510505
  (0, 274)	0.09021200873707368
  (0, 5274)	0.11108562744414445
  (0, 13599)	0.1036413987316636
  (0, 5437)	0.1036413987316636
  (0, 3678)	0.21392179219912877
  (0, 3065)	0.22208377802661425
  (0, 5836)	0.1646750903586285
  (0, 14378)	0.33962752210959823
  (0, 16587)	0.12549432354918996
  (0, 3225)	0.24960162956997736
  (0, 14271)	0.21392179219912877
  (0, 4945)	0.24025852494110758
  (0, 15261)	0.07095833561276566
  (0, 16998)	0.1282126322850579
  (0, 11192)	0.09049319826481456
  (0, 11503)	0.27211310056983656
  (0, 13349)	0.15021264094167086
  (0, 17007)	0.23643326319898797
  (0, 17290)	0.20197912553916567
  (0, 13319)	0.2177470539412484
  (0, 14064)	0.20596090415084142
  (0, 16668)	0.19843263965100372
  (0, 14608)	0.15150672398763912
  (0, 8756)	0.22709015857011816
  :	:
  (4801, 403)	0.17727585190343223
  (4801, 4835)	0.247137650

In [19]:
# getting the similarity scores using cosines similarity
similarity=cosine_similarity(feature_vector)

In [20]:
print(similarity)

[[1.         0.07219487 0.037733   ... 0.         0.         0.        ]
 [0.07219487 1.         0.03281499 ... 0.03575545 0.         0.        ]
 [0.037733   0.03281499 1.         ... 0.         0.05389661 0.        ]
 ...
 [0.         0.03575545 0.         ... 1.         0.         0.02651502]
 [0.         0.         0.05389661 ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.02651502 0.         1.        ]]


In [21]:
print(similarity.shape)

(4803, 4803)


In [22]:
# getting tthe movie name from the user

movie_name=input('enter the your favourite movie')

enter the your favourite movie avatar


In [23]:
# creating a list with all movie names givn in the dataset
list_of_movie=movies['title'].tolist()
# print(list_of_movie)

In [24]:
# finding the close match for the movie name given by the user
find_close_matches=difflib.get_close_matches(movie_name,list_of_movie)

In [25]:
print(find_close_matches)

['Avatar']


In [26]:
first_movie=find_close_matches[0]
print(first_movie)

Avatar


In [27]:
index_of_the_movie= movies[movies.title == first_movie]['index'].values[0]
print(index_of_the_movie)

0


In [28]:
# getting a list of similar movie
similarity_score= list(enumerate(similarity[index_of_the_movie]))
# print(similarity_score)

In [29]:
# sorting the movie based on similarity_score
sorted_similar_movie=sorted(similarity_score,key= lambda x:x[1],reverse=True)

In [30]:
# print(sorted_similar_movie)

In [31]:
# print the name of similar movies based on the index
print('movie suggested for you :\n')

movie suggested for you :



In [32]:
i=1
for movie in sorted_similar_movie:
    index=movie[0]
    titel_from_index=movies[movies.index == index]['title'].values[0]
    if(i<20):
        print(i,'. ', titel_from_index)
        i+=1

1 .  Avatar
2 .  Alien
3 .  Aliens
4 .  Guardians of the Galaxy
5 .  Star Trek Beyond
6 .  Star Trek Into Darkness
7 .  Galaxy Quest
8 .  Alien³
9 .  Cargo
10 .  Trekkies
11 .  Gravity
12 .  Moonraker
13 .  Jason X
14 .  Pocahontas
15 .  Space Cowboys
16 .  The Helix... Loaded
17 .  Lockout
18 .  Event Horizon
19 .  Space Dogs


In [33]:
# MOVIE RECOMENDATION SYSTEM

In [34]:
movie_name=input('enter your favourite movie:')

list_of_movie=movies['title'].tolist()

find_close_matches=difflib.get_close_matches(movie_name,list_of_movie)

first_movie=find_close_matches[0]

index_of_the_movie=movies[movies.title == first_movie]['index'].values[0]

similarity_score=list(enumerate(similarity[index_of_the_movie]))

sorted_similar_movie=sorted(similarity_score,key=lambda x:x[1],reverse=True)

print('movie suggested for you :\n')

i=1
for movie in sorted_similar_movie:
    index=movie[0]
    title_from_index=movies[movies.index == index]['title'].values[0]
    if(i<7):
        print(i,'. ', title_from_index)
        i+=1

enter your favourite movie: avatar


movie suggested for you :

1 .  Avatar
2 .  Alien
3 .  Aliens
4 .  Guardians of the Galaxy
5 .  Star Trek Beyond
6 .  Star Trek Into Darkness


In [1]:

!pip freeze > requirements.txt